In [18]:
import os
import getpass
import sqlite3
import uuid
import random
import re
from typing import TypedDict, Optional

from pydantic import BaseModel, Field

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

In [19]:
def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ").strip()

_set_env("GROQ_API_KEY")
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)


In [20]:
def extract_text(agent_resp) -> str:
    """Return the final agent message as a clean string.

    Gemini (and the LangChain 1.0 message format) can return message .content
    as a LIST of content blocks, e.g. [{"type": "text", "text": "..."}],
    instead of a plain string. Calling .strip() on a list raises
    AttributeError, so this normalizes both shapes to a string.
    """
    if isinstance(agent_resp, dict) and "messages" in agent_resp:
        content = agent_resp["messages"][-1].content
    else:
        return str(agent_resp).strip()
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                parts.append(block.get("text", ""))
            elif isinstance(block, str):
                parts.append(block)
        return "".join(parts).strip()

    return str(content).strip()


In [21]:
_UUID_RE = re.compile(
    r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}",
    re.IGNORECASE,
)

def extract_order_id(agent_resp) -> str:
    text = extract_text(agent_resp)
    match = _UUID_RE.search(text)
    return match.group(0) if match else text


In [22]:
def payment_failed(agent_resp) -> bool:
    """Reliably detect a payment failure.

    The payment tool returns a raw "FAIL:..." or "SUCCESS" string, but the LLM
    then paraphrases it ("The payment failed because..."), so checking only the
    final message text is unreliable. We scan ALL messages -- including the raw
    ToolMessage output -- for the failure signal.
    """
    if isinstance(agent_resp, dict) and "messages" in agent_resp:
        for msg in agent_resp["messages"]:
            content = getattr(msg, "content", "")
            if isinstance(content, list):
                content = " ".join(
                    b.get("text", "") if isinstance(b, dict) else str(b)
                    for b in content
                )
            if "FAIL" in str(content).upper():
                return True
        return False
    return "FAIL" in extract_text(agent_resp).upper()

In [23]:
conn = sqlite3.connect("orders.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id TEXT PRIMARY KEY,
    item TEXT,
    amount REAL,
    payment_status TEXT,
    notification_status TEXT
)
""")
conn.commit()

In [24]:

class CreateOrderInput(BaseModel):
    item: str = Field(description="Item being ordered")
    amount: float = Field(description="Order amount")

class PaymentInput(BaseModel):
    order_id: str = Field(description="Order ID for payment")

class NotifyInput(BaseModel):
    order_id: str = Field(description="Order ID to notify")

In [25]:
    
def create_order_tool(data: CreateOrderInput):
    order_id = str(uuid.uuid4())
    cursor.execute(
        "INSERT INTO orders VALUES (?, ?, ?, ?, ?)",
        (order_id, data.item, data.amount, "PENDING", "NOT_SENT")
    )
    conn.commit()
    return {"order_id": order_id}

def process_payment_tool(data: PaymentInput):
    cursor.execute(
        "UPDATE orders SET payment_status = ? WHERE order_id = ?",
        ("SUCCESS", data.order_id)
    )
    conn.commit()
    return {"payment_status": "SUCCESS"}

def notify_user_tool(data: NotifyInput):
    cursor.execute(
        "UPDATE orders SET notification_status = ? WHERE order_id = ?",
        ("SENT", data.order_id)
    )
    conn.commit()
    return {"notification": "Email Sent"}

In [26]:
@tool
def create_order_tool_agent(item: str, amount: float = 100.0) -> str:
    """Create an order and return the generated order_id"""
    result = create_order_tool(CreateOrderInput(item=item, amount=amount))
    return result["order_id"]

@tool
def process_payment_tool_agent(order_id: str) -> str:
    """Process payment for a given order_id"""
    result = process_payment_tool(PaymentInput(order_id=order_id))
    return result["payment_status"]


@tool
def notify_user_tool_agent(order_id: str) -> str:
    """Notify user for a given order_id"""
    result = notify_user_tool(NotifyInput(order_id=order_id))
    return result["notification"]

In [27]:
order_agent = create_agent(
    model=llm,
    tools=[create_order_tool_agent],
    system_prompt="""
You are an Order Service Agent. When given an item name and optional amount,
return the order_id string.
"""
)

payment_agent = create_agent(
    model=llm,
    tools=[process_payment_tool_agent],
    system_prompt="""
You are a Payment Service Agent. When given an order_id, return PAYMENT_SUCCESS or FAIL.
"""
)

notification_agent = create_agent(
    model=llm,
    tools=[notify_user_tool_agent],
    system_prompt="""
You are a Notification Service Agent. When given an order_id, return NOTIFICATION_SENT.
"""
)

In [28]:
class WorkflowState(TypedDict):
    user_input: str
    order_id: Optional[str]
    payment_status: Optional[str]
    notification_status: Optional[str]
    error: Optional[str]
    retries: int

MAX_RETRIES = 2

In [29]:
def order_requester(state: WorkflowState):
    print("[OrderRequester] Running")

    try:
        item = state["user_input"]
        amount = 100.0
        agent_resp = order_agent.invoke({
            "messages": [{"role": "user", "content": f"Create order for {item} with amount {amount}."}]
        })
        order_id = extract_order_id(agent_resp)
        return {
            "order_id": order_id,
            "retries": state["retries"]
        }

    except Exception as e:
        return {"error": str(e), "retries": state["retries"] + 1}

In [30]:
def payment_processor(state: WorkflowState):
    print("[PaymentProcessor] Running")

    try:
        agent_resp = payment_agent.invoke({
            "messages": [{"role": "user", "content": f"Process payment for order {state['order_id']}"}]
        })

        payment_status = extract_text(agent_resp)

        return {
            "payment_status": payment_status,
            "retries": state["retries"]
        }

    except Exception as e:
        return {"error": str(e), "retries": state["retries"] + 1}


In [31]:
def notifier(state: WorkflowState):
    print("[Notifier] Running")

    try:
        agent_resp = notification_agent.invoke({
            "messages": [{"role": "user", "content": f"Notify user for order {state['order_id']}"}]
        })

        notification_status = extract_text(agent_resp)

        return {
            "notification_status": notification_status,
            "retries": state["retries"]
        }

    except Exception as e:
        return {"error": str(e), "retries": state["retries"] + 1}
        

In [32]:
def check_error(state: WorkflowState):
    if state.get("error"):
        if state["retries"] >= MAX_RETRIES:
            print("[X] Max retries reached. Ending workflow.")
            return END
        print("[Retry] Retrying...")
        return "order_requester"
    return "payment_processor"
    
builder = StateGraph(WorkflowState)

builder.add_node("order_requester", order_requester)
builder.add_node("payment_processor", payment_processor)
builder.add_node("notifier", notifier)

builder.set_entry_point("order_requester")

builder.add_conditional_edges("order_requester", check_error)
builder.add_edge("payment_processor", "notifier")
builder.add_edge("notifier", END)

app = builder.compile()

In [35]:
result = app.invoke({
    "user_input": "Laptop Purchase",
    "order_id": None,
    "payment_status": None,
    "notification_status": None,
    "error": None,
    "retries": 0
})




[OrderRequester] Running
[Retry] Retrying...
[OrderRequester] Running
[X] Max retries reached. Ending workflow.


In [36]:
print("\n========== FINAL STATE (BASIC) ==========")
print(result)


========== FINAL STATE (BASIC) ==========
{'user_input': 'Laptop Purchase', 'order_id': None, 'payment_status': None, 'notification_status': None, 'error': 'database is locked', 'retries': 2}
